# Exercise 2.1: Workforce Transformation Readiness - Classification

In this notebook, we consume some SAP-managed ([data products from SuccessFactors](https://api.sap.com/dataproducts?$filter=(LinesOfBusiness:%5B%22Human%20Resources%22%5D))) from the SAP BDC Catalog. The data products are shared to SAP Snowflake for consumption.

> [!IMPORTANT]
> <b>In the notebook, please pay attention to the ✍️ icon. Whenever there is an instruction with the ✍️ icon, edit the code to fit your own user. </b> 


## Task
We use <b>Snowflake Cortex ML.CLASSIFICATION</b> on 8 SAP SuccessFactors data products (which are shared zero-copy from BDC) to predict each employee's transformation-readiness tier. Training labels are bootstrapped from the BDC data itself using business rules, requiring no external datasets.

Classification uses machine learning algorithms to sort data into different classes using patterns detected in training data.  It involves creating a classification model object, passing in a reference to the training data. The model is fitted to the provided training data. You then use the resulting schema-level classification model object to classify new data points and to understand the model’s accuracy through its evaluation APIs.

### Objective
Using the SAP-managed Human Resources data products, train a multi-class classification model that predicts which transformation-readiness tier each employee belongs to:

| Tier | Meaning |
|---|---|
| `READY_NOW` | Fully prepared for the transformation program |
| `FAST_TRACK_3MO` | On a strong trajectory — will be ready within 3 months |
| `NEEDS_COACHING` | Active but stalling — needs targeted coaching |
| `NEEDS_PROGRAM` | Not yet engaged — needs a structured program |

### Data Products Used (all SAP-managed, shared from BDC)
- **Learning:** `LearningEventHistory`, `LearningEnrollment`
- **Succession & Career Development Planning (CDP):** `CareerDevelopmentPlanningData`, `DevelopmentGoals`, `SuccessionReadiness`, `HighPotentialEmployee`
- **Performance & Goals:** `PerformanceRatingStandard`, `GoalStatus`
- **Skills:** `WorkforceSkillsData` *(one engagement signal only — no skill inventory metrics)*
- **Employee Central:** `WorkforcePerson` *(identity / department / job family)*

### No Pre-loaded Datasets
All training data — including the target labels — is **derived from the SAP BDC data products** using domain-expert business rules. No external CSV or synthetic data required.

## Step 0 — Set Context and create schema
✍️ Replace the `<SCHEMANAME>` with your username. Eg. CREATE SCHEMA IF NOT ESISTS TECHED_DA261.TAC12XXXXXXXXX

In [ ]:
--STOP: before running, change the SCHEMA NAME to your USERNAME
USE WAREHOUSE SYSTEM$STREAMLIT_NOTEBOOK_WH;
USE DATABASE TECHED_DA261;
CREATE SCHEMA IF NOT EXISTS TECHED_DA261.<SCHEMANAME> WITH MANAGED ACCESS EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED' CATALOG = 'SNOWFLAKE';

## Step 1 — Explore the SAP BDC Data Products

After sharing from BDC and mounting in Snowflake, each data product appears as a **catalog-linked database**. Let's verify the data is flowing.

In [ ]:
%%sql -r dataframe_2
-- List all mounted databases from BDC
SHOW DATABASES;

In [ ]:
%%sql -r dataframe_3
-- Preview key tables — confirm data is accessible
SELECT * FROM LEARNINGEVENTHISTORY_V1."learningeventhistory"."learningeventhistory" LIMIT 10;
SELECT * FROM SUCCESSIONREADINESS_V1."successionreadiness"."successionreadiness"LIMIT 10;
SELECT * FROM PERFORMANCERATINGSTANDARD_V1."performanceratingstandard"."performanceratingstandard" LIMIT 10;
SELECT * FROM WORKFORCESKILLSDATA_V1."workforceskillsdata"."workforceskillsdata"  LIMIT 10;
SELECT * FROM WORKFORCEPERSON_V1."workforceperson"."person" LIMIT 10;

## Step 2 — Build Feature Tables

Now we transform raw SAP data products into **behavioral engagement indicators** — the features our ML model will learn from.

Each feature tables below queries one or more SAP BDC data products and distills them into a single engagement signal per employee. We are deliberately measuring *what employees are doing* (learning activity, CDP progress, performance trajectory) rather than *what skills they have*.

| Feature Group | Data Products Used | What We Extract |
|---|---|---|
| Learning Behavior | `LearningEventHistory` | Completion rate, learning hours, velocity trend, recency |
| Career Development | `CareerDevelopmentPlanningData`, `DevelopmentGoals` | CDP milestone rate, active goal count |
| Succession Readiness | `SuccessionManagement`, `SuccessionReadiness` | Readiness level, 12-month delta, high-potential flag |
| Performance Trajectory | `PerformanceReviews`, `DevelopmentGoals` | Latest rating, 3-cycle trend, goal completion rate |
| Skills Engagement | `WorkforceSkillsData` | Profile update frequency (engagement signal only) |

> **Why not use skill proficiency or coverage?** Those are descriptive metrics about *current state*. We need *forward-looking behavioral signals* that predict whether someone will be ready for the transformation program.

### 2a. Learning Behavioral Features

**Source:** [`LearningEventHistory`](https://api.sap.com/dataproducts?$filter=(LinesOfBusiness:%5B%22Human%20Resources%22%5D)) from the *SAP SuccessFactors Learning Data Products* package

We extract five engagement indicators from each employee's learning history:

| Feature | Logic | Why It Matters |
|---|---|---|
| `digital_program_completion_rate` | % of items with completion status | Overall program engagement |
| `learning_hours_last_6mo` | Credit hours in the past 6 months | Recent learning intensity |
| `hours_recent_3mo` / `hours_prior_3mo` | Split for velocity calculation | Acceleration vs. deceleration |
| `days_since_last_learning` | Recency of last completion | Flags disengaged employees |

The following table `vw_learning_features` summarises each employee's learning behaviour — how many courses they completed, how many hours they studied recently, whether they're speeding up or slowing down, and how long since they last learned anything. Think of it as a "learning engagement scorecard" per person.


✍️ Replace `<SCHEMANAME>` with your username. Eg. TECHED_DA261.TAC12XXXXXXXXX

In [ ]:

CREATE OR REPLACE ICEBERG TABLE TECHED_DA261.<SCHEMANAME>.vw_learning_features
ICEBERG_VERSION = 3
CATALOG = 'SNOWFLAKE'
EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED'
ICEBERG_MERGE_ON_READ_BEHAVIOR = 'DISABLED'
STORAGE_SERIALIZATION_POLICY = 'COMPATIBLE'
AS
SELECT
    "StudentID"                                             AS employee_id,

    -- Digital program completion rate (%)
    COUNT_IF("ItemCompletionStatusID" LIKE '%COMPL%')
      / NULLIF(COUNT(*), 0) * 100                          AS digital_program_completion_rate,

    -- Learning hours in last 6 months
    SUM(CASE WHEN "CompletionDateTime" >= DATEADD(month, -6, CURRENT_DATE)
             THEN "TotalCreditHours" ELSE 0 END)           AS learning_hours_last_6mo,

    -- Learning velocity: compare last 3 months vs prior 3 months
    SUM(CASE WHEN "CompletionDateTime" >= DATEADD(month, -3, CURRENT_DATE)
             THEN "TotalCreditHours" ELSE 0 END)           AS hours_recent_3mo,
    SUM(CASE WHEN "CompletionDateTime" BETWEEN DATEADD(month, -6, CURRENT_DATE)
                                      AND DATEADD(month, -3, CURRENT_DATE)
             THEN "TotalCreditHours" ELSE 0 END)           AS hours_prior_3mo,

    -- Recency: days since last learning activity
    DATEDIFF(day, MAX("CompletionDateTime"), CURRENT_DATE)  AS days_since_last_learning

FROM LEARNINGEVENTHISTORY_V1."learningeventhistory"."learningeventhistory"
GROUP BY "StudentID";

### 2b. Career Development & Succession Features

**Sources:** `CareerDevelopmentPlanningData`, `DevelopmentGoals`, `SuccessionManagement`, `SuccessionReadiness` from the *SAP SuccessFactors Succession and Career Development Data Products* package

Two tables here — one for CDP/goals, one for succession readiness:

**CDP Features** (first code cell below):
- `cdp_milestone_completion_rate` — % of development goals marked *Completed*
- `active_dev_goal_count` — goals in *On Track*, *At Risk*, *Behind*, or *Not Started*

**Succession Features** (second code cell below):
- `current_readiness_level` — highest readiness rank from succession nominations
- `readiness_level_delta_12mo` — how much readiness has changed (max − min rank)
- `high_potential_flag` — 1 if nominee has a *"Ready Now"* readiness label

> **Data flow:** `SuccessionManagement` provides the nominee records; `SuccessionReadiness` provides the readiness code/label lookup. They are joined on `readinessValue = readinessCode`.

The following table `vw_cdp_features` captures each employee's career development plan progress — what percentage of their development milestones they've finished, and how many active goals they still have open. It answers: "Is this person actively working on their career growth?"

✍️ Replace `<SCHEMANAME>` with your username. Eg. TECHED_DA261.TAC12XXXXXXXXX

In [ ]:
CREATE OR REPLACE ICEBERG TABLE TECHED_DA261.<SCHEMANAME>.vw_cdp_features
ICEBERG_VERSION = 3
CATALOG = 'SNOWFLAKE'
EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED'
ICEBERG_MERGE_ON_READ_BEHAVIOR = 'DISABLED'
STORAGE_SERIALIZATION_POLICY = 'COMPATIBLE'
AS
SELECT
    cdp."userId"                                             AS employee_id,
    COUNT_IF(dg."state" = 'Completed')
      / NULLIF(COUNT(dg."id"), 0) * 100                      AS cdp_milestone_completion_rate,
    COUNT_IF(dg."state" IN ('On Track','At Risk','Behind','Not Started'))
                                                              AS active_dev_goal_count
FROM CAREERDEVELOPMENTPLANNINGDATA_V1."careerdevelopmentplanningdata"."careerdevelopmentplanningdata" cdp
LEFT JOIN DEVELOPMENTGOALS_V1."developmentgoals"."developmentgoal" dg
    ON cdp."userId" = dg."userId"
GROUP BY cdp."userId"

The following table `vw_succession_features` tracks how ready an employee is to step into a bigger role based on succession planning data — their readiness level, how much that readiness has improved over the past year, and whether they've been flagged as high-potential. It's the "bench strength" indicator.

✍️ Replace `<SCHEMANAME>` with your username. Eg. TECHED_DA261.TAC12XXXXXXXXX

In [ ]:
CREATE OR REPLACE ICEBERG TABLE TECHED_DA261.<SCHEMANAME>.vw_succession_features
ICEBERG_VERSION = 3
CATALOG = 'SNOWFLAKE'
EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED'
ICEBERG_MERGE_ON_READ_BEHAVIOR = 'DISABLED'
STORAGE_SERIALIZATION_POLICY = 'COMPATIBLE'
AS
SELECT
    n."nomineeUserId"                                     AS employee_id,
    MAX(sr."readinessRank")                               AS current_readiness_level,
    -- Readiness change across nominations
    MAX(sr."readinessRank") - MIN(sr."readinessRank")     AS readiness_level_delta_12mo,
    -- High-potential flag: nominee with "Ready Now" readiness (code 3.0)
    MAX(CASE WHEN sr."readinessLabel" = 'Ready Now' THEN 1 ELSE 0 END)
                                                          AS high_potential_flag
FROM SUCCESSIONMANAGEMENT_V1."successionmanagement"."nominee" n
LEFT JOIN SUCCESSIONREADINESS_V1."successionreadiness"."successionreadiness" sr
    ON n."readinessValue" = sr."readinessCode"
WHERE n."deleted" = FALSE
GROUP BY n."nomineeUserId";

### 2c. Performance & Goals Features

**Sources:** `PerformanceReviews` from the *SAP SuccessFactors Performance and Goals Data Products* package, plus `DevelopmentGoals` (reused from 2b)

We compute a 3-cycle performance trajectory plus a goal completion rate:

| Feature | Logic | Why It Matters |
|---|---|---|
| `latest_perf_rating` | Most recent review rating | Current performance level |
| `perf_rating_delta` | Latest rating minus 3rd-latest rating | Improving, stable, or declining? |
| `goal_completion_rate` | % of development goals completed | Follow-through on commitments |


The following table `vw_performance_features` pulls together each employee's performance review history — their latest rating, whether their ratings are trending up or down over the last 3 review cycles, and what percentage of development goals they've completed. It shows: "Is this person performing and following through?"

✍️ Replace  `<SCHEMANAME>` with your username. Eg. TECHED_DA261.TAC12XXXXXXXXX

In [ ]:
CREATE OR REPLACE ICEBERG TABLE TECHED_DA261.<SCHEMANAME>.vw_performance_features
ICEBERG_VERSION = 3
CATALOG = 'SNOWFLAKE'
EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED'
ICEBERG_MERGE_ON_READ_BEHAVIOR = 'DISABLED'
STORAGE_SERIALIZATION_POLICY = 'COMPATIBLE'
AS
WITH ranked_ratings AS (
    SELECT "subject" AS employee_id, "rating" AS rating_value, "reviewEnd" AS review_end_date,
           ROW_NUMBER() OVER (PARTITION BY "subject"
                              ORDER BY "reviewEnd" DESC) AS cycle_rank
    FROM PERFORMANCEREVIEWS_V1."performancereviews"."form"
    WHERE "type" = 'Review'
),
goal_stats AS (
    SELECT "userId" AS employee_id,
           COUNT_IF("state" = 'Completed')                AS goals_completed,
           COUNT(*)                                       AS goals_total
    FROM DEVELOPMENTGOALS_V1."developmentgoals"."developmentgoal"
    GROUP BY "userId"
)
SELECT
    r.employee_id,
    MAX(CASE WHEN r.cycle_rank = 1 THEN r.rating_value END)   AS latest_perf_rating,
    -- Performance trend: latest rating minus 3rd-latest rating
    MAX(CASE WHEN r.cycle_rank = 1 THEN r.rating_value END)
    - MAX(CASE WHEN r.cycle_rank = 3 THEN r.rating_value END) AS perf_rating_delta,
    -- Goal completion rate
    MAX(g.goals_completed) / NULLIF(MAX(g.goals_total), 0) * 100
                                                               AS goal_completion_rate
FROM ranked_ratings r
LEFT JOIN goal_stats g ON r.employee_id = g.employee_id
WHERE r.cycle_rank <= 3
GROUP BY r.employee_id;

### 2d. Skills Engagement Signal

**Source:** `WorkforceSkillsData` from the *SAP SuccessFactors Skills Data Products* package

We extract **one signal only**: how often the employee has updated their skill profile in the last 6 months.

- `skill_profile_updates_6mo` — count of skills assigned/updated recently

This is a *behavioral engagement signal* — employees actively maintaining their profile are more likely to be engaged in the transformation program. We deliberately do **not** extract:
- Skill coverage or gap counts
- Proficiency scores
- Skill-to-role match rates

Those are descriptive *Skills Insights* metrics — not predictive engagement indicators.

The following table `vw_skills_signal` counts how many times an employee updated their skills profile in the last 6 months. It's a simple engagement signal — people who bother keeping their profile current are more likely to be engaged in the transformation program.

✍️ Replace `<SCHEMANAME>` with your username. Eg. TECHED_DA261.TAC12XXXXXXXXX

In [ ]:
-- We use ONLY the update frequency as a program engagement signal.
-- No skill gap counts, no coverage rates, no proficiency scores.
CREATE OR REPLACE ICEBERG TABLE TECHED_DA261.<SCHEMANAME>.vw_skills_signal
ICEBERG_VERSION = 3
CATALOG = 'SNOWFLAKE'
EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED'
ICEBERG_MERGE_ON_READ_BEHAVIOR = 'DISABLED'
STORAGE_SERIALIZATION_POLICY = 'COMPATIBLE'
AS
SELECT
    "personUuid"                                              AS employee_id,
    COUNT_IF("skillAssignedDate" >= DATEADD(month, -6, CURRENT_DATE))
                                                              AS skill_profile_updates_6mo
FROM WORKFORCESKILLSDATA_V1."workforceskillsdata"."skillsdata"
GROUP BY "personUuid";

## Step 3 — Generate Training Labels from BDC Data

Supervised ML needs labeled training data — but we don't have an external CSV of "who is ready." Instead, we use **time-shifted labeling**: features are computed from data *before* a cutoff date (T = 2024-06-30), and labels are derived from *outcomes observed after* that cutoff (T to T+12 months).

This is a rigorous technique that avoids data leakage — the model learns from past behavior to predict future outcomes.

### Time Windows

| Window | Date Range | Used For |
|---|---|---|
| Feature window | Up to 2024-06-30 | All 12 behavioral features (learning, CDP, succession, performance, skills) |
| Label window | 2024-07-01 to 2025-06-30 | Observed outcomes → tier assignment |

### Labeling Rules (applied to outcomes in the label window)

| Tier | Rule |
|---|---|
| `READY_NOW` | ≥ 5 future completions AND future performance rating ≥ 4 |
| `FAST_TRACK_3MO` | ≥ 4 future completions AND future performance ≥ 3 |
| `NEEDS_COACHING` | ≥ 1 future completion OR ≥ 1 future skill update |
| `NEEDS_PROGRAM` | Everyone else |

> **In production**, these labels would come from manager assessments or program milestone sign-offs. For this hands-on lab, we derive them entirely from BDC data — no external files needed.

The following table `VW_LABELED_EMPLOYEES` is the master training dataset for the ML model. It combines all 5 feature tables above into one row per employee, then assigns each person a readiness label (READY_NOW, FAST_TRACK, NEEDS_COACHING, or NEEDS_PROGRAM) based on what they actually did in the following 12 months. This is what the AI learns from.

✍️ Replace `<SCHEMANAME>` with your username. Eg. TECHED_DA261.TAC12XXXXXXXXX

In [ ]:
-- Time-shifted labeling: features from T (<=2024-06-30), labels from outcomes T to T+12mo
-- Features and labels are independent — no data leakage
CREATE OR REPLACE ICEBERG TABLE TECHED_DA261.<SCHEMANAME>.VW_LABELED_EMPLOYEES
ICEBERG_VERSION = 3
CATALOG = 'SNOWFLAKE'
EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED'
ICEBERG_MERGE_ON_READ_BEHAVIOR = 'DISABLED'
STORAGE_SERIALIZATION_POLICY = 'COMPATIBLE'
AS
WITH time_windows AS (
    SELECT '2024-06-30'::DATE AS feature_cutoff,
           '2024-07-01'::DATE AS label_start,
           '2025-06-30'::DATE AS label_end
),

-- Employee population: anyone with activity before the feature cutoff
employee_population AS (
    SELECT DISTINCT employee_id
    FROM (
        SELECT "StudentID" AS employee_id FROM LEARNINGEVENTHISTORY_V1."learningeventhistory"."learningeventhistory"
               WHERE "CompletionDateTime" <= (SELECT feature_cutoff FROM time_windows)
        UNION
        SELECT "userId" FROM CAREERDEVELOPMENTPLANNINGDATA_V1."careerdevelopmentplanningdata"."careerdevelopmentplanningdata"
        UNION
        SELECT "nomineeUserId" FROM SUCCESSIONMANAGEMENT_V1."successionmanagement"."nominee" WHERE "deleted" = FALSE
        UNION
        SELECT "subject" FROM PERFORMANCEREVIEWS_V1."performancereviews"."form"
               WHERE "type" = 'Review' AND "reviewEnd" <= (SELECT feature_cutoff FROM time_windows)
    )
),

------------------------------------------------------------------------
-- FEATURES: measured using data UP TO feature_cutoff (T = 2024-06-30)
------------------------------------------------------------------------
lf AS (
    SELECT
        "StudentID" AS employee_id,
        COUNT_IF("ItemCompletionStatusID" IN ('COURSE-COMPL','ONLINE-COMPL','MOOC-CMPL'))
          / NULLIF(COUNT(*), 0) * 100                        AS digital_program_completion_rate,
        SUM(CASE WHEN "CompletionDateTime" >= DATEADD(month, -6, (SELECT feature_cutoff FROM time_windows))
                 THEN "TotalCreditHours" ELSE 0 END)         AS learning_hours_last_6mo,
        SUM(CASE WHEN "CompletionDateTime" >= DATEADD(month, -3, (SELECT feature_cutoff FROM time_windows))
                 THEN "TotalCreditHours" ELSE 0 END)         AS hours_recent_3mo,
        SUM(CASE WHEN "CompletionDateTime" BETWEEN DATEADD(month, -6, (SELECT feature_cutoff FROM time_windows))
                                          AND     DATEADD(month, -3, (SELECT feature_cutoff FROM time_windows))
                 THEN "TotalCreditHours" ELSE 0 END)         AS hours_prior_3mo,
        DATEDIFF(day, MAX("CompletionDateTime"),
                 (SELECT feature_cutoff FROM time_windows))   AS days_since_last_learning
    FROM LEARNINGEVENTHISTORY_V1."learningeventhistory"."learningeventhistory"
    WHERE "CompletionDateTime" <= (SELECT feature_cutoff FROM time_windows)
    GROUP BY "StudentID"
),
cf AS (
    SELECT
        cdp."userId" AS employee_id,
        COUNT_IF(dg."state" = 'Completed')
          / NULLIF(COUNT(dg."id"), 0) * 100                  AS cdp_milestone_completion_rate,
        COUNT_IF(dg."state" IN ('On Track','At Risk','Behind','Not Started'))
                                                              AS active_dev_goal_count
    FROM CAREERDEVELOPMENTPLANNINGDATA_V1."careerdevelopmentplanningdata"."careerdevelopmentplanningdata" cdp
    LEFT JOIN DEVELOPMENTGOALS_V1."developmentgoals"."developmentgoal" dg
        ON cdp."userId" = dg."userId"
    GROUP BY cdp."userId"
),
sf AS (
    SELECT
        n."nomineeUserId" AS employee_id,
        MAX(sr."readinessRank")                               AS current_readiness_level,
        MAX(sr."readinessRank") - MIN(sr."readinessRank")     AS readiness_level_delta_12mo,
        MAX(CASE WHEN sr."readinessLabel" = 'Ready Now' THEN 1 ELSE 0 END) AS high_potential_flag
    FROM SUCCESSIONMANAGEMENT_V1."successionmanagement"."nominee" n
    LEFT JOIN SUCCESSIONREADINESS_V1."successionreadiness"."successionreadiness" sr
        ON n."readinessValue" = sr."readinessCode"
    WHERE n."deleted" = FALSE
    GROUP BY n."nomineeUserId"
),
ranked_ratings AS (
    SELECT "subject" AS employee_id, "rating" AS rating_value, "reviewEnd" AS review_end_date,
           ROW_NUMBER() OVER (PARTITION BY "subject" ORDER BY "reviewEnd" DESC) AS cycle_rank
    FROM PERFORMANCEREVIEWS_V1."performancereviews"."form"
    WHERE "type" = 'Review'
      AND "reviewEnd" <= (SELECT feature_cutoff FROM time_windows)
),
goal_stats AS (
    SELECT "userId" AS employee_id,
           COUNT_IF("state" = 'Completed') AS goals_completed,
           COUNT(*) AS goals_total
    FROM DEVELOPMENTGOALS_V1."developmentgoals"."developmentgoal"
    GROUP BY "userId"
),
pf AS (
    SELECT
        r.employee_id,
        MAX(CASE WHEN r.cycle_rank = 1 THEN r.rating_value END) AS latest_perf_rating,
        MAX(CASE WHEN r.cycle_rank = 1 THEN r.rating_value END)
        - MAX(CASE WHEN r.cycle_rank = 3 THEN r.rating_value END) AS perf_rating_delta,
        MAX(g.goals_completed) / NULLIF(MAX(g.goals_total), 0) * 100 AS goal_completion_rate
    FROM ranked_ratings r
    LEFT JOIN goal_stats g ON r.employee_id = g.employee_id
    WHERE r.cycle_rank <= 3
    GROUP BY r.employee_id
),
sk AS (
    SELECT
        "personUuid" AS employee_id,
        COUNT_IF("skillAssignedDate" >= DATEADD(month, -6, (SELECT feature_cutoff FROM time_windows))
                 AND "skillAssignedDate" <= (SELECT feature_cutoff FROM time_windows)) AS skill_profile_updates_6mo
    FROM WORKFORCESKILLSDATA_V1."workforceskillsdata"."skillsdata"
    GROUP BY "personUuid"
),

------------------------------------------------------------------------
-- LABELS: derived from OUTCOMES in label window (T to T+12mo)
------------------------------------------------------------------------
future_learning AS (
    SELECT "StudentID" AS employee_id,
           SUM("TotalCreditHours")   AS future_learning_hours,
           COUNT_IF("ItemCompletionStatusID" IN ('COURSE-COMPL','ONLINE-COMPL','MOOC-CMPL')) AS future_completions
    FROM LEARNINGEVENTHISTORY_V1."learningeventhistory"."learningeventhistory"
    WHERE "CompletionDateTime" BETWEEN (SELECT label_start FROM time_windows)
                                   AND (SELECT label_end   FROM time_windows)
    GROUP BY "StudentID"
),
future_perf AS (
    SELECT "subject" AS employee_id,
           MAX("rating") AS future_perf_rating
    FROM PERFORMANCEREVIEWS_V1."performancereviews"."form"
    WHERE "type" = 'Review'
      AND "reviewEnd" BETWEEN (SELECT label_start FROM time_windows)
                          AND (SELECT label_end   FROM time_windows)
    GROUP BY "subject"
),
future_skills AS (
    SELECT "personUuid" AS employee_id,
           COUNT(*) AS future_skill_updates
    FROM WORKFORCESKILLSDATA_V1."workforceskillsdata"."skillsdata"
    WHERE "skillAssignedDate" BETWEEN (SELECT label_start FROM time_windows)
                                 AND (SELECT label_end   FROM time_windows)
    GROUP BY "personUuid"
)

------------------------------------------------------------------------
-- FINAL: features from T, labels from T+12mo outcomes
------------------------------------------------------------------------
SELECT
    wp.employee_id,

    -- Features (all measured at or before feature_cutoff)
    COALESCE(lf.digital_program_completion_rate, 0)       AS digital_program_completion_rate,
    COALESCE(lf.learning_hours_last_6mo, 0)               AS learning_hours_last_6mo,
    CASE
        WHEN COALESCE(lf.hours_prior_3mo, 0) = 0 THEN 'STABLE'
        WHEN lf.hours_recent_3mo > lf.hours_prior_3mo * 1.15 THEN 'ACCELERATING'
        WHEN lf.hours_recent_3mo < lf.hours_prior_3mo * 0.85 THEN 'DECELERATING'
        ELSE 'STABLE'
    END                                                   AS learning_velocity_trend,
    COALESCE(lf.days_since_last_learning, 999)            AS days_since_last_learning,
    COALESCE(cf.cdp_milestone_completion_rate, 0)         AS cdp_milestone_completion_rate,
    COALESCE(cf.active_dev_goal_count, 0)                 AS active_dev_goal_count,
    COALESCE(sf.readiness_level_delta_12mo, 0)            AS readiness_level_delta_12mo,
    COALESCE(sf.high_potential_flag, 0)                    AS high_potential_flag,
    COALESCE(pf.latest_perf_rating, 0)                    AS latest_perf_rating,
    COALESCE(pf.perf_rating_delta, 0)                     AS perf_rating_delta,
    COALESCE(pf.goal_completion_rate, 0)                  AS goal_completion_rate,
    COALESCE(sk.skill_profile_updates_6mo, 0)             AS skill_profile_updates_6mo,

    -- Labels (derived from outcomes AFTER the feature cutoff)
    CASE
        WHEN COALESCE(fl.future_completions, 0) >= 5
         AND COALESCE(fp.future_perf_rating, 0) >= 4
            THEN 'READY_NOW'
        WHEN COALESCE(fl.future_completions, 0) >= 4
         AND COALESCE(fp.future_perf_rating, 0) >= 3
            THEN 'FAST_TRACK_3MO'
        WHEN COALESCE(fl.future_completions, 0) >= 1
          OR COALESCE(fs.future_skill_updates, 0) >= 1
            THEN 'NEEDS_COACHING'
        ELSE 'NEEDS_PROGRAM'
    END                                                   AS readiness_tier_label

FROM employee_population wp
LEFT JOIN lf ON wp.employee_id = lf.employee_id
LEFT JOIN cf ON wp.employee_id = cf.employee_id
LEFT JOIN sf ON wp.employee_id = sf.employee_id
LEFT JOIN pf ON wp.employee_id = pf.employee_id
LEFT JOIN sk ON wp.employee_id = sk.employee_id
LEFT JOIN future_learning fl ON wp.employee_id = fl.employee_id
LEFT JOIN future_perf     fp ON wp.employee_id = fp.employee_id
LEFT JOIN future_skills   fs ON wp.employee_id = fs.employee_id;

✍️ Replace `<SCHEMANAME>` with your username. Eg. TECHED_DA261.TAC12XXXXXXXXX

In [ ]:
-- Check label distribution — should have a reasonable spread
SELECT readiness_tier_label, COUNT(*) AS cnt,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
FROM TECHED_DA261.<SCHEMANAME>.VW_LABELED_EMPLOYEES
GROUP BY 1
ORDER BY 2 DESC;

## Step 4 — Train the Classification Model

One SQL statement is all it takes. [`SNOWFLAKE.ML.CLASSIFICATION`](https://docs.snowflake.com/en/user-guide/ml-functions/classification) handles everything under the hood:

- **Automatic train/test split** — no manual holdout setup
- **Algorithm selection** — Snowflake tests multiple algorithms and picks the best
- **Hyperparameter tuning** — automatic cross-validation
- **Evaluation metrics** — built-in precision, recall, F1 per class (enabled via `evaluate: TRUE`)

The model takes 12 numeric/categorical features as input and predicts `READINESS_TIER_LABEL` — one of the four tiers defined in Step 3.

✍️ Replace `<SCHEMANAME>` with your username. Eg. TECHED_DA261.TAC12XXXXXXXXX. 

In [ ]:
-- Train multi-class classification model using SNOWFLAKE.ML.CLASSIFICATION
-- Input: labeled employees table with 13 features + readiness_tier_label target
-- evaluate=TRUE enables built-in evaluation metrics (precision, recall, F1)
CREATE OR REPLACE SNOWFLAKE.ML.CLASSIFICATION TECHED_DA261.<SCHEMANAME>.readiness_velocity_model(
    INPUT_DATA     => SYSTEM$REFERENCE('TABLE', 'TECHED_DA261.<SCHEMANAME>.VW_LABELED_EMPLOYEES'),
    TARGET_COLNAME => 'READINESS_TIER_LABEL',
    CONFIG_OBJECT  => {'evaluate': TRUE}
);

## Step 5 — Evaluate the Model

Three built-in evaluation calls:

### 5a. Evaluation Metrics
Per-class **precision**, **recall**, and **F1-score**, plus overall **accuracy**. This tells us how reliably the model distinguishes each tier.

### 5b. Confusion Matrix
Shows where the model gets confused — e.g., does it mix up `FAST_TRACK_3MO` with `NEEDS_COACHING`? This is critical for understanding model quality before deploying to production.

### 5c. Feature Importance
This is the **most valuable output for the business**. It answers: *which behavioral engagement indicators are the strongest predictors of transformation readiness?* The CHRO can use this to prioritize L&D investments — e.g., if `learning_velocity_trend` ranks highest, accelerating learning programs will have the most impact.


✍️ Replace `<SCHEMANAME>` with your username. Eg. TECHED_DA261.TAC12XXXXXXXXX

In [ ]:
-- Show per-class precision, recall, F1-score, and overall accuracy
CALL TECHED_DA261.<SCHEMANAME>.readiness_velocity_model!SHOW_EVALUATION_METRICS();

✍️ Replace `<SCHEMANAME>` with your username. Eg. TECHED_DA261.TAC12XXXXXXXXX

In [ ]:
-- Show predicted vs actual tier counts — identifies which tiers the model confuses
CALL TECHED_DA261.<SCHEMANAME>.readiness_velocity_model!SHOW_CONFUSION_MATRIX();

✍️ Replace `<SCHEMANAME>` with your username. Eg. TECHED_DA261.TAC12XXXXXXXXX

In [ ]:
-- Show which features drive the classification — learning hours, velocity, CDP rate, etc.
CALL TECHED_DA261.<SCHEMANAME>.readiness_velocity_model!SHOW_FEATURE_IMPORTANCE();

✍️ Replace `<SCHEMANAME>` with your username. Eg. TECHED_DA261.TAC12XXXXXXXXX

In [ ]:
import snowflake.snowpark.context as ctx
import matplotlib.pyplot as plt

session = ctx.get_active_session()
df = session.sql('CALL TECHED_DA261.<SCHEMANAME>.readiness_velocity_model!SHOW_FEATURE_IMPORTANCE()').to_pandas()

df.columns = [c.strip('"') for c in df.columns]
top = df.nlargest(10, 'SCORE')

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top['FEATURE'][::-1], top['SCORE'][::-1], color='#0070F2')
ax.set_xlabel('Importance Score')
ax.set_title('Top 10 Feature Importance — What Drives Transformation Readiness?')
plt.tight_layout()
plt.show()

## Step 6 — Score the Current Workforce

Now we apply the trained model to **every employee** in the dataset. For each person, the model returns:

| Output | Description |
|---|---|
| `readiness_tier` | Predicted tier (READY_NOW / FAST_TRACK_3MO / NEEDS_COACHING / NEEDS_PROGRAM) |
| `p_ready_now`, `p_fast_track`, etc. | Per-class probability (how confident the model is) |
| `readiness_velocity_index` | Weighted composite score (0–100) — higher = more ready |

The **velocity index** is a weighted sum of tier probabilities:
```
velocity = (P(READY_NOW) × 100) + (P(FAST_TRACK) × 72) + (P(COACHING) × 38) + (P(PROGRAM) × 12)
```

We also join with **WorkforcePerson** (`Employee Central`) to enrich each row with `department` and `job_title`.

The output table `workforce_readiness_scores` will be used later for the forecast. It contains the ML model's predictions for every employee — which readiness tier they belong to, how confident the model is about each tier, and a 0–100 velocity index score. Also includes department and job title for dashboard filtering. 

✍️ Replace `<SCHEMANAME>` with your username. Eg. TECHED_DA261.TAC12XXXXXXXXX

In [ ]:
-- Score every employee in the labeled dataset using the trained model
-- Produces: readiness_tier (predicted class), per-class probabilities,
-- and readiness_velocity_index (weighted composite score 0-100)
-- Joins with Employee Central (WorkforcePerson) for department and job title
CREATE OR REPLACE ICEBERG TABLE TECHED_DA261.<SCHEMANAME>.workforce_readiness_scores
ICEBERG_VERSION = 3
CATALOG = 'SNOWFLAKE'
EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED'
ICEBERG_MERGE_ON_READ_BEHAVIOR = 'DISABLED'
STORAGE_SERIALIZATION_POLICY = 'COMPATIBLE'
AS
WITH scored AS (
    SELECT
        employee_id,
        TECHED_DA261.<SCHEMANAME>.readiness_velocity_model!PREDICT(
            OBJECT_CONSTRUCT(*)
        )                                                     AS prediction,
        prediction:class::STRING                              AS readiness_tier,
        prediction['probability']['READY_NOW']::FLOAT         AS p_ready_now,
        prediction['probability']['FAST_TRACK_3MO']::FLOAT    AS p_fast_track,
        prediction['probability']['NEEDS_COACHING']::FLOAT    AS p_needs_coaching,
        prediction['probability']['NEEDS_PROGRAM']::FLOAT     AS p_needs_program,
        -- Readiness velocity index: weighted sum of tier probabilities
        -- Weights: READY_NOW=100, FAST_TRACK=72, NEEDS_COACHING=38, NEEDS_PROGRAM=12
        ROUND(
            (prediction['probability']['READY_NOW']::FLOAT       * 100) +
            (prediction['probability']['FAST_TRACK_3MO']::FLOAT  *  72) +
            (prediction['probability']['NEEDS_COACHING']::FLOAT  *  38) +
            (prediction['probability']['NEEDS_PROGRAM']::FLOAT   *  12),
        1)                                                    AS readiness_velocity_index
    FROM TECHED_DA261.<SCHEMANAME>.VW_LABELED_EMPLOYEES
),
emp_dept AS (
    -- Get latest department and job title from Employee Central
    SELECT DISTINCT p."externalId" AS employee_id,
           j."department" AS department, j."jobTitle" AS job_title
    FROM WORKFORCEPERSON_V1."workforceperson"."person" p
    JOIN WORKFORCEPERSON_V1."workforceperson"."assignment" a ON p."id" = a."personId"
    JOIN WORKFORCEPERSON_V1."workforceperson"."jobdetail" j ON a."id" = j."_id"
    WHERE j."department" IS NOT NULL
    QUALIFY ROW_NUMBER() OVER (PARTITION BY p."externalId" ORDER BY j."startDate" DESC) = 1
)
SELECT s.employee_id, ed.department, ed.job_title,
       s.readiness_tier, s.p_ready_now, s.p_fast_track, s.p_needs_coaching, s.p_needs_program,
       s.readiness_velocity_index, s.prediction
FROM scored s
LEFT JOIN emp_dept ed ON s.employee_id = ed.employee_id;

## Step 7 — OPTIONAL: Explore the Results 

Three exploration queries plus a visualization:

1. **Tier distribution** — overall split across the four tiers with average velocity index
2. **Tier breakdown with percentages** — same data, formatted as % of total workforce
3. **Top 20 at-risk employees** — lowest velocity index, enriched with department and job title from Employee Central
4. **Stacked bar chart** — visual breakdown of readiness tiers by department (top 15)

> **What to look for:**
> - What % of the workforce is in `NEEDS_PROGRAM`? That's your transformation risk exposure.
> - Which departments have the highest concentration of `NEEDS_COACHING`? Those need targeted interventions.
> - Are there departments where nearly everyone is `READY_NOW`? Those are your transformation champions.

✍️ Replace `<SCHEMANAME>` with your username. Eg. TECHED_DA261.TAC12XXXXXXXXX

In [ ]:
-- Readiness tier distribution
SELECT readiness_tier, COUNT(*) AS employee_count,
       ROUND(AVG(readiness_velocity_index), 1) AS avg_velocity
FROM TECHED_DA261.<SCHEMANAME>.workforce_readiness_scores
GROUP BY 1
ORDER BY avg_velocity DESC;

✍️ Replace `<SCHEMANAME>` with your username. Eg. TECHED_DA261.TAC12XXXXXXXXX

In [ ]:
-- Readiness tier distribution (department not available without Employee Central)
SELECT readiness_tier, COUNT(*) AS cnt,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct,
       ROUND(AVG(readiness_velocity_index), 1) AS avg_velocity
FROM TECHED_DA261.<SCHEMANAME>.workforce_readiness_scores
GROUP BY 1
ORDER BY avg_velocity DESC;

✍️ Replace `<SCHEMANAME>` with your username. Eg. TECHED_DA261.TAC12XXXXXXXXX

In [ ]:
-- Top 20 most at-risk employees (with department from Employee Central)
WITH emp_dept AS (
    SELECT DISTINCT p."externalId" AS employee_id, j."department" AS department, j."jobTitle" AS job_title
    FROM WORKFORCEPERSON_V1."workforceperson"."person" p
    JOIN WORKFORCEPERSON_V1."workforceperson"."assignment" a ON p."id" = a."personId"
    JOIN WORKFORCEPERSON_V1."workforceperson"."jobdetail" j ON a."id" = j."_id"
    WHERE j."department" IS NOT NULL
    QUALIFY ROW_NUMBER() OVER (PARTITION BY p."externalId" ORDER BY j."startDate" DESC) = 1
)
SELECT ws.employee_id, ed.department, ed.job_title,
       ws.readiness_tier, ws.readiness_velocity_index
FROM   TECHED_DA261.<SCHEMANAME>.workforce_readiness_scores ws
LEFT JOIN emp_dept ed ON ws.employee_id = ed.employee_id
ORDER BY ws.readiness_velocity_index ASC
LIMIT 20;

✍️ Replace `<SCHEMANAME>` with your username. Eg. TECHED_DA261.TAC12XXXXXXXXX

In [ ]:
import snowflake.snowpark.context as ctx
import matplotlib.pyplot as plt

session = ctx.get_active_session()
df = session.sql('''

    SELECT ws.department, ws.readiness_tier, COUNT(*) AS cnt
    FROM  TECHED_DA261.<SCHEMANAME>.workforce_readiness_scores  ws
    GROUP BY 1, 2
    ORDER BY 1, 2
''').to_pandas()

pivot = df.pivot(index='DEPARTMENT', columns='READINESS_TIER', values='CNT').fillna(0)
colors = {'READY_NOW': '#107E3E', 'FAST_TRACK_3MO': '#0070F2',
          'NEEDS_COACHING': '#F0AB00', 'NEEDS_PROGRAM': '#E76500'}

top_depts = pivot.sum(axis=1).nlargest(15).index
pivot_top = pivot.loc[top_depts]

pivot_top.plot(kind='barh', stacked=True, figsize=(12, 6),
               color=[colors.get(c, '#999') for c in pivot_top.columns])
plt.xlabel('Employee Count')
plt.title('Readiness Tier Distribution by Department (Top 15)')
plt.legend(title='Tier', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Done!

Our seven persisted tables are now ready. All feature tables and output tables created in this notebook are created as **Iceberg tables** . However, they can also be created as **Dynamic Iceberg tables** that auto-refresh within 1 hour. 

The syntax to create it is as follows:

CREATE OR REPLACE DYNAMIC ICEBERG TABLE DATABASE.SCHEMA.TABLENAME_1
  TARGET_LAG = '1 hour'
  WAREHOUSE = DEMOWAREHOUSE
  EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED'
  CATALOG = 'SNOWFLAKE'
  ICEBERG_MERGE_ON_READ_BEHAVIOR = 'DISABLED'
  STORAGE_SERIALIZATION_POLICY = 'COMPATIBLE'
AS SELECT COLUMNS from DATABASE.SCHEMA.TABLENAME_2

**A note on table type: Iceberg Tables vs. Dynamic Iceberg Tables:**

In this exercise we create standard Iceberg Tables rather than Dynamic Iceberg Tables. Here's why:

Dynamic Iceberg Tables automatically materialize and refresh their results based on a defined target lag. This means Snowflake continuously runs a background refresh process that consumes compute resources (virtual warehouse credits) to keep the table in sync with its upstream sources. While this is powerful for near-real-time pipelines, it is not required for a TechEd Hands-on.

Now, proceed to Notebook 2 to build the 12-month department readiness trajectory forecast and publish the combined custom data product to the BDC catalog.